# 1. What & Why Feature Selection?

### Concept & Definition
Feature selection identifies and retains the most informative variables while removing irrelevant or redundant features.

### Impact of Excessive / Irrelevant Features:
1. **Curse of Dimensionality:** Increases search space exponentially, leading to overfitting.
2. **Computational Inefficiency:** Slower model training and inference latency.
3. **Multicollinearity:** Destroys model interpretability in linear classifiers.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("Cleaned_Validated_Data.csv")

# Create numeric target
df["Target"] = np.where(df["Churn"] == "Yes", 1, 0)
X_num = df[["Age", "TenureYears", "MonthlyCharges"]].copy().fillna(0)
y = df["Target"]

print(f"Feature matrix shape: {X_num.shape}")

Feature matrix shape: (1010, 3)


# 2. Variance Thresholding

### Concept & Definition
A baseline filter method that removes all features whose variance does not meet a specified threshold (e.g., quasi-constant or constant features).

In [2]:
from sklearn.feature_selection import VarianceThreshold

# Add dummy zero-variance feature
X_test_vt = X_num.copy()
X_test_vt["Constant_Col"] = 1.0

vt = VarianceThreshold(threshold=0.01)
X_vt_selected = vt.fit_transform(X_test_vt)

print("Original Feature Count:", X_test_vt.shape[1])
print("Post VarianceThreshold Count:", X_vt_selected.shape[1])

Original Feature Count: 4
Post VarianceThreshold Count: 3


# 3. Correlation-Based Selection

### Concept & Definition
Identifies pairs of continuous features with high Pearson correlation coefficients ($|r| > 0.85$) and drops one to remove redundancy (multicollinearity).

In [3]:
# Correlation matrix check
corr_matrix = X_num.corr().abs()

# Identify collinear features
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.85)]

print("Highly Collinear Features to Drop:", to_drop)

Highly Collinear Features to Drop: []


# 4. Univariate Selection & Mutual Information

### Concepts:
- **ANOVA F-Test:** Measures linear dependency between continuous features and categorical target.
- **Mutual Information:** Measures non-linear dependency based on information gain.

In [4]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

# Mutual Information Scoring
mi_scores = mutual_info_classif(X_num, y, random_state=42)
mi_df = pd.DataFrame({"Feature": X_num.columns, "MI_Score": mi_scores}).sort_values(by="MI_Score", ascending=False)

print("=== Mutual Information Scores ===")
display(mi_df)

=== Mutual Information Scores ===


,Feature,MI_Score
0,Age,0.008195
1,TenureYears,0.000000
2,MonthlyCharges,0.000000


# 5. Recursive Feature Elimination (RFE) & Tree Importance

### Concepts:
- **RFE:** Fits a model recursively and drops the least important features step-by-step.
- **Tree-Based Importance:** Extracts feature importance scores directly from ensembles (Random Forest / XGBoost).

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE

# Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_num, y)

# RFE
rfe = RFE(estimator=rf, n_features_to_select=2)
rfe.fit(X_num, y)

rfe_summary = pd.DataFrame({
    "Feature": X_num.columns,
    "RF Importance": rf.feature_importances_.round(4),
    "RFE Selected": rfe.support_
})

display(rfe_summary)
print("Notebook 10 execution completed successfully!")

,Feature,RF Importance,RFE Selected
0,Age,0.4479,True
1,TenureYears,0.3052,True
2,MonthlyCharges,0.2469,False


Notebook 10 execution completed successfully!
